[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baluragala/building-rag-pipelines/blob/main/notebooks/05_augmentation_generation.ipynb)

# Building RAG Pipelines
## Notebook 05: Augmentation & Generation
**Duration:** 40 min &nbsp;|&nbsp; **Mode:** Conceptual + Guided Coding &nbsp;|&nbsp; upGrad Live Session

> Taught **WHY → WHAT → HOW**. We keep asking *"What happens if this step is poorly
> designed?"* and we **predict before we run** and **compare outputs**. LangChain is
> shown as a **parallel mapping** — it abstracts mechanics but not design decisions.

![pipeline](https://dummyimage.com/1000x70/1f2937/ffffff&text=Loading+%E2%86%92+Chunking+%E2%86%92+Retrieval+%E2%86%92+Augmentation+%E2%86%92+Generation+%E2%86%92+Evaluation)

In [ ]:
# ============================================================
# COLAB BOOTSTRAP — run this cell first. (Same as every notebook.)
# ============================================================
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/baluragala/building-rag-pipelines.git"  # INSTRUCTOR: set this

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

_pip("numpy", "openai", "tiktoken", "rank-bm25", "beautifulsoup4", "pypdf",
     "langchain-community", "langchain-text-splitters", "langchain-openai", "faiss-cpu")
try:
    import rag_pipeline
except ModuleNotFoundError:
    if IN_COLAB:
        subprocess.run(["git", "clone", "-q", REPO_URL], check=False)
        if os.path.isdir("building-rag-pipelines"):
            sys.path.insert(0, "building-rag-pipelines")
        else:
            print("Clone failed. Upload `rag_pipeline/` + `data/` via the Colab file browser, then re-run.")
    else:
        sys.path.insert(0, os.path.abspath(".."))
    import rag_pipeline

def data_path(*parts):
    for base in ("data", "../data", "building-rag-pipelines/data"):
        p = os.path.join(base, *parts)
        if os.path.exists(p):
            return p
    return os.path.join("data", *parts)

print("rag_pipeline", rag_pipeline.__version__, "ready.  Colab:", IN_COLAB)

In [ ]:
# Providers: OpenAI by default; offline MOCK if no key (class always runs).
import os
if not os.getenv("OPENAI_API_KEY"):
    os.environ["RAG_LLM_PROVIDER"] = "mock"
    os.environ["RAG_EMBED_PROVIDER"] = "mock"
# For the real stack: set OPENAI_API_KEY (getpass or Colab userdata) BEFORE this cell.
from rag_pipeline import config
print(config.current_config())

In [ ]:
from rag_pipeline.loaders import load_directory
docs = load_directory(data_path("corpus"))
print(f"Loaded {len(docs)} documents from the Acme Cloud corpus.")

## WHY — the stage everyone conflates with retrieval

Name the distinction sharply:
- **Retrieval = WHICH chunks** the system finds.
- **Augmentation = HOW those chunks are packed into the prompt** the LLM reads.
- **Generation = the LLM turning that prompt into a grounded, cited answer.**

> **What happens if augmentation is poorly designed?** You blow the context budget
> (cost + latency), or you bury the answer in the middle of a long context where
> LLMs attend to it least ("lost in the middle"), or you hand the model unlabelled
> text it can't cite.
>
> **What happens if generation is poorly designed?** The model ignores the context
> and answers from memory (hallucination), pads with fluent filler, or refuses to
> say "I don't know" when it should. Most of these are **prompt** failures, not
> model failures — which is why prompt design *is* a RAG design decision.

## WHAT — three context-injection strategies

| Strategy | Idea | Trade-off |
|----------|------|-----------|
| **Naive stuffing** | concatenate all chunks into one prompt | simple; fails when chunks exceed the window or dilute attention |
| **Map-reduce** | answer from each chunk (map), then combine (reduce) | scales to many chunks; more LLM calls |
| **Refine** | seed from one chunk, iteratively revise as each further chunk is read | good for cumulative reasoning; sequential, slowest |

**DO** control context size; structure the prompt; label each chunk `[n]` with its
source. **DON'T** dump everything and hope; **DON'T** drop provenance.

**Safe RAG prompting:** instruction ("use ONLY the context"), **grounding**
("answer 'I don't know' if the context is insufficient" — this single line kills a
lot of hallucination), and **citations** (`[n]` tags tied to numbered chunks).

In [ ]:
from rag_pipeline import config
from rag_pipeline.chunking import recursive_chunk
from rag_pipeline.vectorstore import InMemoryVectorStore
from rag_pipeline.embeddings import embed_documents
from rag_pipeline.retrieval import DenseRetriever, BM25Retriever, HybridRetriever
from rag_pipeline.augmentation import stuff, map_reduce, refine, format_context, collect_sources

emb, llm = config.get_embedder(), config.get_llm()
chunks = recursive_chunk(docs, 500, 50)
store  = InMemoryVectorStore().add(chunks, embed_documents(emb, chunks))
hybrid = HybridRetriever(DenseRetriever(store, emb), BM25Retriever(chunks))
print("Index ready.")

### The context block — provenance is built in

`format_context` labels each chunk `[1] (source, page/chunk)`, respecting a token
budget, so the generation prompt can *require* `[n]` citations. This is what makes
answers auditable.

In [ ]:
q = "Which plan should I choose if I need SSO and a 99.99% SLA?"   # multi-hop
hits = hybrid.retrieve(q, k=4)
print(format_context(hits, max_tokens=1200)[:700])

> ### ✋ Predict before you run
> The question above needs evidence from MULTIPLE documents (SSO is in security, the 99.99% SLA is in the SLA doc, the plan name is in pricing). Which injection strategy — stuff, map-reduce, or refine — do you expect to handle this multi-document synthesis best, and why?
>
> *Commit to a guess before executing. Comparing prediction vs result is the point.*

In [ ]:
# Compare context-injection methods on the multi-hop question.
print("=== STUFF ===")
from rag_pipeline.generation import generate_answer
print(generate_answer(llm, q, stuff(hits, max_tokens=1200))[:400])

print("\n=== MAP-REDUCE ===")
print(map_reduce(llm, q, hits)[:400])

print("\n=== REFINE ===")
print(refine(llm, q, hits)[:400])

*(With the offline mock LLM the answers are honest extractive stubs; with a real
OpenAI/Claude key you'll see genuinely different synthesis quality across the
three strategies — run it both ways if you have a key.)*

## HOW — grounding & citations, and the hallucination demo

Now the headline experiment. We ask a question the corpus **cannot** answer
("Does Acme offer a native mobile app?") with two prompts:
- **grounded** (our safe RAG template: use only context, else say "I don't know", cite `[n]`),
- **ungrounded** (a weak template that lets the model answer however it likes).

**Predict:** which one hallucinates?

In [ ]:
from rag_pipeline.generation import generate_answer, answer_with_sources
neg_q = "Does Acme Cloud offer a native mobile app for iOS and Android?"
context = stuff(hybrid.retrieve(neg_q, k=4), max_tokens=1200)

print("--- GROUNDED prompt (should REFUSE) ---")
print(generate_answer(llm, neg_q, context, grounded=True))
print("\n--- UNGROUNDED prompt (may INVENT an answer) ---")
print(generate_answer(llm, neg_q, context, grounded=False))

**What you should observe (with a real LLM):** the grounded prompt answers *"I
don't know based on the provided context"*, while the ungrounded prompt tends to
**invent** plausible mobile-app details. Same model, same context — the *prompt
design* is the difference. That is the agenda's "compare outputs with/without
structured prompts", and it's why grounding is non-negotiable.

In [ ]:
# The production entry point: retrieve -> (rerank) -> format -> generate, WITH sources.
res = answer_with_sources(llm, hybrid, "How are webhook payloads signed?", k=3)
print("ANSWER:", res["answer"][:300])
print("\nSOURCES (returned for auditability):")
for s in res["sources"]:
    print("  -", s["label"], f"score={s['score']}")

## HOW (parallel mapping) — LangChain LCEL chain

`{context, question} | prompt | ChatOpenAI | StrOutputParser` is the same
retrieve→prompt→generate flow. Crucially, the chain encodes the **same** grounding
and citation rules we wrote by hand — the framework does **not** decide them for you.

In [ ]:
try:
    from rag_pipeline.retrieval import build_langchain_retriever
    from rag_pipeline.generation import build_rag_chain_langchain
    chain = build_rag_chain_langchain(build_langchain_retriever(chunks, k=3))
    print("LCEL RAG chain built:", type(chain).__name__)
    # print(chain.invoke("How much is the Growth plan?"))   # needs OpenAI key
except Exception as e:
    print("LangChain not installed — concept still holds:", e)

## Recap
- Retrieval picks chunks; **augmentation** packs them; **generation** answers.
- stuff / map-reduce / refine trade simplicity vs scale vs cumulative reasoning.
- **Grounding + citations** turn a fluent guesser into a trustworthy, auditable system.

**Next → Notebook 06 (Evaluation):** how do we *know* any of this is working? Metrics,
faithfulness, RAGAS, and building an eval dataset.